# Main-and-Transfer Substation

The main-and-transfer configuration adds a transfer bus beside the main bus, linked by a bus-tie
breaker. Normally every branch is fed from the main bus through its own breaker. To maintain a branch
breaker without taking the branch out, the branch's normally-open bypass disconnector is closed onto
the transfer bus, and the bus-tie breaker protects the branch in its place.

![Main-and-Transfer Substation](../images/main_transfer.png)

## CIM Representation

The main and transfer buses are `ConnectivityNode`s with `BusbarSection`s, joined by a tie bay. Each
branch is a breaker bay from the main bus to the branch node, plus a normally-open bypass
`Disconnector` from the branch node to the transfer bus.

| Object | mRID |
|---|---|
| Main bus, transfer bus | `vl_115_bus_main`, `vl_115_bus_transfer` (BusbarSections `..._bbs`) |
| Bus tie | `vl_115_tie_d1`, `vl_115_tie_brk`, `vl_115_tie_d2` |
| Branch *n* | `vl_115_bus_main` - `vl_115_n_d1` - `vl_115_n_brk` - `vl_115_n_d2` - branch node |
| Branch *n* bypass (normally open) | branch node - `vl_115_n_bypass` - `vl_115_bus_transfer` |

The diagram labels come from an earlier release of CIM-Builder. The table gives the mRIDs the builder writes now, all of them prefixed with the VoltageLevel mRID `vl_115`. Every node and switch goes in the `VoltageLevel` and takes its `BaseVoltage`.

----

## Build the Substation

### Import CIMantic Graphs, the CIM profile and the builders

The CIM profile and unit handling are process-wide environment variables, so set them before importing CIMantic Graphs.

In [1]:
import logging
import os

os.environ["CIMG_CIM_PROFILE"] = "cim18v17"
os.environ["CIMG_USE_UNITS"] = "TRUE"
os.environ["CIMG_SYSTEM_MVA_BASE"] = "100"
# The IEEE 13 files carry CIMHub extensions that cim18v17 does not have: hide the parser warnings
logging.getLogger("cimgraph").setLevel(logging.ERROR)

import cimgraph.utils as utils
import cimprofiles.cim18.cim18v17 as cim
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
from cimtables.units import Qty

from cimbuilder.object_builder import BaseVoltageBuilder, FeederBuilder
from cimbuilder.substation_builder import MainAndTransferSubstationBuilder

### Load the IEEE 13 and IEEE 13 Assets feeders

Both feeder files load into one graph. The substation is built into the same graph, so that its switches connect to the feeders' existing `sourcebus` nodes. Each feeder's head node is the `ConnectivityNode` named `sourcebus` inside that feeder.

In [2]:
IEEE13 = "49AD8E07-3BF9-A4E2-CB8F-C3722F837B62"
IEEE13_ASSETS = "5B816B93-7A5F-B64C-8460-47C17D6E4B0F"

feeder_files = ["../../tests/test_models/IEEE13.xml", "../../tests/test_models/IEEE13_Assets.xml"]
network = FeederModel(
    container=cim.Feeder(mRID=IEEE13), connection=XMLFile(filename=feeder_files), distributed=False
)
container = network.container

In [3]:
def sourcebus(network, feeder_mRID):
    """mRID of the feeder head node, the ConnectivityNode named sourcebus in the feeder"""
    for node in network.graph[cim.ConnectivityNode].values():
        node_container = node.ConnectivityNodeContainer
        if node.name == "sourcebus" and node_container is not None and node_container.mRID == feeder_mRID:
            return node.mRID
    raise ValueError(f"no sourcebus in feeder {feeder_mRID}")


print("IEEE 13 sourcebus:       ", sourcebus(network, IEEE13))
print("IEEE 13 Assets sourcebus:", sourcebus(network, IEEE13_ASSETS))

IEEE 13 sourcebus:        A8A25B50-3AE3-4A31-A18B-B3FA13397ED3
IEEE 13 Assets sourcebus: E14BD0DF-466D-487B-8786-0D46515D1F9B


### Create the Substation and its 115 kV VoltageLevel

References between objects are mRID strings, as in every other builder. The `BaseVoltage` is created first so the VoltageLevel can refer to it.

In [4]:
BaseVoltageBuilder(network, container).create(
    mRID="bv_115kV", name="115 kV", nominalVoltage=Qty(115, "kV")
)

builder = MainAndTransferSubstationBuilder(network, container)
substation = builder.create(mRID="sub_main_and_transfer", name="sub_main_and_transfer")
substation, voltage_level = builder.add_voltage_level(
    substation, mRID="vl_115", name="115 kV", BaseVoltage="bv_115kV"
)

### Add the buses

`add_buses` adds the main bus, the transfer bus and the bus tie between them.

In [5]:
def show(switches):
    """one line per switch: class, mRID, the nodes on its two terminals and normalOpen"""
    for switch in switches:
        node1, node2 = [terminal.ConnectivityNode for terminal in switch.Terminals]
        print(
            f"{type(switch).__name__:13} {switch.mRID:24} "
            f"{node1.name or node1.mRID} - {node2.name or node2.mRID}   normalOpen={switch.normalOpen}"
        )

In [6]:
voltage_level, busbars, ties = builder.add_buses(voltage_level)
print([busbar.mRID for busbar in busbars])
show(ties)

['vl_115_bus_main_bbs', 'vl_115_bus_transfer_bbs']
Disconnector  vl_115_tie_d1            vl_115_bus_main - vl_115_tie_j1   normalOpen=False
Breaker       vl_115_tie_brk           vl_115_tie_j1 - vl_115_tie_j2   normalOpen=False
Disconnector  vl_115_tie_d2            vl_115_tie_j2 - vl_115_bus_transfer   normalOpen=False


### Add the branches

Each branch returns its three bay switches followed by its bypass disconnector.

In [7]:
voltage_level, switches = builder.add_branch(voltage_level, number=1, node="line_1")
show(switches)
voltage_level, switches = builder.add_branch(voltage_level, number=2, node=sourcebus(network, IEEE13))
show(switches)
voltage_level, switches = builder.add_branch(voltage_level, number=3, node=sourcebus(network, IEEE13_ASSETS))
show(switches)

Disconnector  vl_115_1_d1              vl_115_bus_main - vl_115_1_j1   normalOpen=False
Breaker       vl_115_1_brk             vl_115_1_j1 - vl_115_1_j2   normalOpen=False
Disconnector  vl_115_1_d2              vl_115_1_j2 - line_1   normalOpen=False
Disconnector  vl_115_1_bypass          line_1 - vl_115_bus_transfer   normalOpen=True
Disconnector  vl_115_2_d1              vl_115_bus_main - vl_115_2_j1   normalOpen=False
Breaker       vl_115_2_brk             vl_115_2_j1 - vl_115_2_j2   normalOpen=False
Disconnector  vl_115_2_d2              vl_115_2_j2 - sourcebus   normalOpen=False
Disconnector  vl_115_2_bypass          sourcebus - vl_115_bus_transfer   normalOpen=True
Disconnector  vl_115_3_d1              vl_115_bus_main - vl_115_3_j1   normalOpen=False
Breaker       vl_115_3_brk             vl_115_3_j1 - vl_115_3_j2   normalOpen=False
Disconnector  vl_115_3_d2              vl_115_3_j2 - sourcebus   normalOpen=False
Disconnector  vl_115_3_bypass          sourcebus - vl_115_bus_tran

### Link the feeders to the substation

The switches connect to the feeders electrically. The feeder container links to the substation through `Feeder.NormalEnergizingSubstation`:

In [8]:
feeder_builder = FeederBuilder(network, container)
for feeder_mRID in (IEEE13, IEEE13_ASSETS):
    feeder = network.find_by_mRID(feeder_mRID)
    feeder_builder.add_connectivity(feeder, NormalEnergizingSubstation=substation.mRID)

print([feeder.name for feeder in substation.NormalEnergizedFeeder])

['ieee13nodeckt', 'ieee13nodecktassets']


### Save the substation and feeders to XML

In [9]:
os.makedirs("output", exist_ok=True)
utils.write_xml(network, "output/main_and_transfer.xml")

----

## Round-Trip Test: Read the XML File Back

Reload the file into a new model and check that the substation, its feeders and its switching equipment all came back. The feeders' total load comes back with them.

In [10]:
roundtrip = FeederModel(
    container=cim.Feeder(mRID=IEEE13),
    connection=XMLFile(filename="output/main_and_transfer.xml"),
    distributed=False,
)
substation = roundtrip.find_by_mRID("sub_main_and_transfer")
voltage_level = roundtrip.find_by_mRID("vl_115")
print("feeders:", [feeder.name for feeder in substation.NormalEnergizedFeeder])
for cim_class in (cim.BusbarSection, cim.Breaker, cim.Disconnector):
    count = sum(isinstance(equipment, cim_class) for equipment in voltage_level.Equipments)
    print(f"{cim_class.__name__}: {count}")

feeders: ['ieee13nodeckt', 'ieee13nodecktassets']
BusbarSection: 2
Breaker: 4
Disconnector: 11


In [11]:
# CIM XML values are in SI base units (W), CIMUnit converts them for display
total_load = sum(float(load.p) for load in roundtrip.graph[cim.EnergyConsumer].values())
print(f"total load served by the substation: {cim.ActivePower(total_load).to('kW'):.1f} kW")

total load served by the substation: 6937.0 kW
